In [ ]:
import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PositionalSharding
from functools import partial
import time

# --- Configuration (Optimized for Stability & Speed)
MAX_RECURSION_DEPTH = 1_000_000  # 🔥 Testing the highest recursion depth
OPTIMAL_DEPTH_STEP = 250_000  # 🔥 Breaking it into manageable steps
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE = 50_000_000  # 🔥 Extreme scaling with 50M samples per batch

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PositionalSharding
from functools import partial
import time

# --- Configuration (Optimized for Stability & Speed)
MAX_RECURSION_DEPTH = 1_000_000  # 🔥 Testing the highest recursion depth
OPTIMAL_DEPTH_STEP = 250_000     # 🔥 Breaking it into manageable steps
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE = 50_000_000          # 🔥 Extreme scaling with 50M samples per batch

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    """Normalizes depth scaling to prevent instability."""
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    """🔥 Executes in optimized recursion chunks to maximize efficiency"""
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))

    def body_fn(i, val):
        pi_dyn = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val

    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# --- 8-Core CPU Sharding Setup ---
# Request CPU devices explicitly and build an 8-core mesh.
devices = jax.devices("cpu")
if len(devices) < 8:
    # If fewer than 8 physical devices are exposed, create a virtual mesh of 8 devices.
    devices = mesh_utils.create_device_mesh((8,))
else:
    # Otherwise, use the first 8 devices.
    devices = devices[:8]

mesh = Mesh(devices, ("data",))
sharding = PositionalSharding(mesh.devices.flat)

batch_input = jnp.linspace(0, 10, BATCH_SIZE)
batch_input = jax.device_put(batch_input, sharding)

batched_dppu_processing = pjit(
    lambda arr: vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=OPTIMAL_DEPTH_STEP, scale_factor=0.5), in_axes=0)(arr),
    in_shardings=(sharding,),
    out_shardings=sharding,
)

# ✅ **Optimized Adaptive Execution**
def process_with_larger_depths(x, total_depth):
    """🔥 Instead of running all at once, we now execute in 250K-depth steps"""
    iterations = total_depth // OPTIMAL_DEPTH_STEP
    for _ in range(iterations):
        x = dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP)
    return x

# --- 🚀 Optimized Execution ---
for depth in [250_000, 500_000, 1_000_000]:
    start_time = time.time()
    output_batch = process_with_larger_depths(batch_input, depth)
    end_time = time.time()
    print(f"✅ Batch Output Shape (Depth={depth}):", output_batch.shape)
    print(f"🔥 Execution Time: {end_time - start_time:.6f} sec")

NUM_TRIALS = 2  # 🔥 Reduce trials to avoid unnecessary overload

# Warm-up compile
_ = dppu_with_dynamic_pi_phi(jnp.ones((BATCH_SIZE,)), depth=OPTIMAL_DEPTH_STEP)

for depth in [250_000, 500_000, 1_000_000]:
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        result = process_with_larger_depths(jnp.ones((BATCH_SIZE,)), depth)
        _ = jax.device_get(result)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    print(f"\n🔥 CPU Benchmark (Depth={depth}, Batch={BATCH_SIZE})")
    print(f"Avg: {avg_time:.6f}, Min: {min(times):.6f}, Max: {max(times):.6f}")

# --- Investigate Compilation Stability ---
compiled_fn_250k = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=250_000)
compiled_fn_1M = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=1_000_000)

print("\n🚀 XLA Compilation for Depth=250,000:")
print(compiled_fn_250k.as_text())

print("\n🚀 XLA Compilation for Depth=1,000,000:")
print(compiled_fn_1M.as_text())

✅ Batch Output Shape (Depth=250000): (50000000,)
🔥 Execution Time: 0.287392 sec
✅ Batch Output Shape (Depth=500000): (50000000,)
🔥 Execution Time: 0.001721 sec
✅ Batch Output Shape (Depth=1000000): (50000000,)
🔥 Execution Time: 0.001507 sec

🔥 CPU Benchmark (Depth=250000, Batch=50000000)
Avg: 251.267066, Min: 129.762367, Max: 372.771764
